# Aspect-Based Sentiment Analysis — Multi-Dataset
### DistilBERT + BIO Tagging + Aspect-Aware Attention
Supports: SemEval 2014 REST/Laptop, SemEval 2015 REST, SemEval 2016 REST, MAMS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch transformers scikit-learn pandas spacy seqeval
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=4e50cdd20080d659bd1f01a37486affc8a48a4614e372c3c227995f161a3fe81
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## ⚙️ Configuration — Set Your Dataset Paths Here

In [ ]:
import os

# ---------------------------------------------------------------
# DATASET PATHS — paste your Google Drive paths here.
# Add or remove entries depending on which datasets you have.
# Supported formats: SemEval 2014, 2015, 2016 (REST or Laptop), MAMS ATSA
# ---------------------------------------------------------------
DATASET_PATHS = [
    # SemEval 2016 Restaurant (your original dataset)
    "/content/drive/MyDrive/absa_data/ABSA16_Restaurants_Train_SB1_v2.xml",

    # SemEval 2015 Restaurant — paste path if you have it
    "/content/drive/MyDrive/absa_data_final/ABSA-15_Restaurants_Train_Final.xml",

    # SemEval 2014 Restaurant — paste path if you have it
     "/content/drive/MyDrive/absa_data_final/Restaurants_Train.xml",

    # SemEval 2014 Laptop — paste path if you have it
    # "/content/drive/MyDrive/absa_data/SemEval14_Laptops_Train.xml",

    # MAMS ATSA Train — paste path if you have it
    "/content/drive/MyDrive/absa_data_final/train.xml",
]

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from transformers import DistilBertTokenizerFast, DistilBertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import xml.etree.ElementTree as ET
import pickle
import numpy as np
from collections import defaultdict, Counter
from sklearn.metrics import classification_report
from seqeval.metrics import classification_report as seq_report
import spacy

nlp = spacy.load('en_core_web_sm')
print('Libraries loaded.')

Libraries loaded.


In [ ]:
class Config:
    MAX_LEN       = 128
    BATCH_SIZE    = 16
    EPOCHS_EXTRACTOR = 5
    EPOCHS_SENTIMENT = 5
    LEARNING_RATE = 2e-5
    WARMUP_RATIO  = 0.1
    TRAIN_RATIO   = 0.8   # 80% train
    VAL_RATIO     = 0.1   # 10% validation
    TEST_RATIO    = 0.1   # 10% test
    DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TAG2ID        = {'O': 0, 'B-ASPECT': 1, 'I-ASPECT': 2}
    ID2TAG        = {v: k for k, v in {'O': 0, 'B-ASPECT': 1, 'I-ASPECT': 2}.items()}
    NUM_TAGS      = 3
    SENTIMENT2ID  = {'positive': 0, 'negative': 1, 'neutral': 2}
    ID2SENTIMENT  = {0: 'positive', 1: 'negative', 2: 'neutral'}
    MAP_PATH      = '/content/term_to_category.pkl'
    EXT_PATH      = '/content/aspect_extractor.pt'
    SENT_PATH     = '/content/sentiment_model.pt'

cfg = Config()
print(f'Device: {cfg.DEVICE}')

Device: cuda


## 🔍 Format-Aware XML Parser
Auto-detects SemEval 2014/MAMS format vs SemEval 2015/2016 format.

In [ ]:
def detect_format(root):
    """
    Detect XML format version.
    '2016': SemEval 2015/2016 — <Review> wrapper, <Opinions>/<Opinion target= category= from= to=>
    '2014': SemEval 2014 / MAMS — no <Review>, <aspectTerms>/<aspectTerm term= from= to=>
    """
    if root.find('Review') is not None:
        return '2016'
    return '2014'


def iter_sentences(root, fmt):
    """
    Yield (sentence_element) for every sentence in the XML,
    regardless of format.
    """
    if fmt == '2016':
        for review in root.findall('Review'):
            for sent in review.findall('sentences/sentence'):
                yield sent
    else:  # 2014 / MAMS
        for sent in root.findall('sentence'):
            yield sent


def get_spans(sent, fmt):
    """
    Extract (from_char, to_char, term) tuples from a sentence element.
    Skips NULL targets and entries with missing offsets.
    """
    spans = []
    if fmt == '2016':
        parent = sent.find('Opinions')
        tags = parent.findall('Opinion') if parent is not None else sent.findall('Opinion')
        for op in tags:
            target = op.get('target')
            if not target or target == 'NULL':
                continue
            frm = op.get('from')
            to  = op.get('to')
            if frm is None or to is None:
                continue
            spans.append((int(frm), int(to), target))
    else:  # 2014 / MAMS
        parent = sent.find('aspectTerms')
        if parent is None:
            return spans
        for at in parent.findall('aspectTerm'):
            term = at.get('term')
            if not term or term == 'NULL':
                continue
            frm = at.get('from')
            to  = at.get('to')
            if frm is None or to is None:
                continue
            spans.append((int(frm), int(to), term))
    return spans


def get_opinions_for_sentiment(sent, fmt):
    """
    Yield (aspect_signal, polarity) pairs for sentiment training.
    For 2016 format: aspect_signal = category (e.g. FOOD#QUALITY)
    For 2014 format: aspect_signal = term itself (no linked category exists)
    """
    if fmt == '2016':
        parent = sent.find('Opinions')
        tags = parent.findall('Opinion') if parent is not None else sent.findall('Opinion')
        for op in tags:
            category = op.get('category')
            polarity = op.get('polarity')
            if category and polarity:
                yield category, polarity
    else:  # 2014 / MAMS
        parent = sent.find('aspectTerms')
        if parent is None:
            return
        for at in parent.findall('aspectTerm'):
            term     = at.get('term')
            polarity = at.get('polarity')
            if term and polarity:
                yield term, polarity


print('Parser functions defined.')

Parser functions defined.


## 📦 Dataset Classes

In [ ]:
def align_bio_tags_with_tokens(text, spans, tokenizer, max_len):
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=max_len,
        return_offsets_mapping=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'][0]
    attention_mask = encoding['attention_mask'][0]
    offset_mapping = encoding['offset_mapping'][0]

    tag_ids = torch.full_like(input_ids, -100, dtype=torch.long)

    for i, (s, e) in enumerate(offset_mapping):
        if s != 0 or e != 0:
            tag_ids[i] = 0  # real token → O

    for start_char, end_char, _ in spans:
        inside = False
        for i, (tok_s, tok_e) in enumerate(offset_mapping):
            if tok_s == 0 and tok_e == 0:
                continue
            if tok_s >= start_char and tok_e <= end_char:
                tag_ids[i] = cfg.TAG2ID['B-ASPECT'] if not inside else cfg.TAG2ID['I-ASPECT']
                inside = True
            else:
                inside = False

    tag_ids = torch.where(attention_mask == 0, torch.tensor(-100), tag_ids)
    return tag_ids, input_ids, attention_mask


class AspectExtractionDataset(Dataset):
    """
    Loads aspect term extraction examples from one XML file.
    Auto-detects SemEval 2014 or 2015/2016 format.
    """
    def __init__(self, xml_file, tokenizer, max_len):
        self.examples = []
        tree = ET.parse(xml_file)
        root = tree.getroot()
        fmt  = detect_format(root)
        skipped = 0
        for sent in iter_sentences(root, fmt):
            text_el = sent.find('text')
            if text_el is None or not text_el.text:
                continue
            text  = text_el.text
            spans = get_spans(sent, fmt)
            if not spans:
                skipped += 1
                continue
            tag_ids, input_ids, attn_mask = align_bio_tags_with_tokens(
                text, spans, tokenizer, max_len
            )
            self.examples.append({
                'input_ids': input_ids,
                'attention_mask': attn_mask,
                'tag_ids': tag_ids
            })
        print(f'  [{fmt}] {xml_file.split("/")[-1]}: {len(self.examples)} examples loaded, {skipped} skipped (no spans)')

    def __len__(self):  return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


class SentimentDataset(Dataset):
    """
    Loads sentiment classification examples from one XML file.
    Auto-detects format. Skips 'conflict' polarity labels.
    """
    def __init__(self, xml_file, tokenizer, max_len):
        self.examples = []
        tree = ET.parse(xml_file)
        root = tree.getroot()
        fmt  = detect_format(root)
        for sent in iter_sentences(root, fmt):
            text_el = sent.find('text')
            if text_el is None or not text_el.text:
                continue
            text = text_el.text
            for aspect_signal, polarity in get_opinions_for_sentiment(sent, fmt):
                if polarity not in cfg.SENTIMENT2ID:
                    continue  # skip 'conflict'
                self.examples.append({
                    'sentence': text,
                    'aspect':   aspect_signal,
                    'label':    cfg.SENTIMENT2ID[polarity]
                })
        print(f'  [{fmt}] {xml_file.split("/")[-1]}: {len(self.examples)} sentiment examples')

    def __len__(self):  return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


print('Dataset classes defined.')

Dataset classes defined.


## 🧠 Model Definitions

In [ ]:
class AspectExtractorDistilBERT(nn.Module):
    """DistilBERT token classifier for BIO aspect extraction."""
    def __init__(self, num_tags):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.distilbert.config.hidden_size, num_tags)
        self.loss_fn    = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, tag_ids=None):
        out    = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.classifier(self.dropout(out.last_hidden_state))
        if tag_ids is not None:
            active  = attention_mask.view(-1) == 1
            loss    = self.loss_fn(
                logits.view(-1, logits.shape[-1])[active],
                tag_ids.view(-1)[active]
            )
            return loss, logits
        return logits


class AspectAwareDistilBERT(nn.Module):
    """DistilBERT + multi-head cross-attention for aspect-conditioned sentiment."""
    def __init__(self, num_labels):
        super().__init__()
        self.distilbert      = DistilBertModel.from_pretrained('distilbert-base-uncased')
        h                    = self.distilbert.config.hidden_size
        self.aspect_attn     = nn.MultiheadAttention(embed_dim=h, num_heads=8,
                                                      batch_first=True, dropout=0.1)
        self.dropout         = nn.Dropout(0.1)
        self.classifier      = nn.Linear(h, num_labels)

    def forward(self, input_ids, attention_mask, aspect_mask):
        seq = self.distilbert(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state
        # Average aspect token embeddings → query
        exp = aspect_mask.unsqueeze(-1).float()
        asp_emb = (seq * exp).sum(1) / (aspect_mask.sum(1, keepdim=True).float() + 1e-8)
        asp_emb = asp_emb.unsqueeze(1)  # (B,1,H)
        # Cross-attention over full sequence
        attn_out, _ = self.aspect_attn(
            query=asp_emb, key=seq, value=seq,
            key_padding_mask=(attention_mask == 0)
        )
        return self.classifier(self.dropout(attn_out.squeeze(1)))


print('Models defined.')

Models defined.


## 🗺️ Term-to-Category Map + Tokenizer

In [ ]:
def build_term_category_map(xml_paths, output_path):
    """Build term→category lookup from all provided dataset files."""
    term_to_cats = defaultdict(list)
    for path in xml_paths:
        if not os.path.exists(path):
            print(f'  Skipping (not found): {path}')
            continue
        tree = ET.parse(path)
        root = tree.getroot()
        fmt  = detect_format(root)
        for sent in iter_sentences(root, fmt):
            if fmt == '2016':
                parent = sent.find('Opinions')
                tags   = parent.findall('Opinion') if parent is not None else []
                for op in tags:
                    tgt = op.get('target')
                    cat = op.get('category')
                    if tgt and tgt != 'NULL' and cat:
                        term_to_cats[tgt.lower()].append(cat)
            else:
                # For 2014, map term → term (used as its own signal)
                parent = sent.find('aspectTerms')
                if parent is not None:
                    for at in parent.findall('aspectTerm'):
                        term = at.get('term')
                        if term and term != 'NULL':
                            term_to_cats[term.lower()].append(term.lower())

    term_to_cat = {t: Counter(cs).most_common(1)[0][0] for t, cs in term_to_cats.items()}
    all_cats    = [c for cs in term_to_cats.values() for c in cs]
    default     = Counter(all_cats).most_common(1)[0][0] if all_cats else 'FOOD#QUALITY'
    term_to_cat['_DEFAULT_'] = default

    with open(output_path, 'wb') as f:
        pickle.dump(term_to_cat, f)
    print(f'Map built: {len(term_to_cat)-1} terms. Default: "{default}"')
    return term_to_cat


tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print('Tokenizer loaded.')

# Build or load term-to-category map
if not os.path.exists(cfg.MAP_PATH):
    term_to_cat = build_term_category_map(DATASET_PATHS, cfg.MAP_PATH)
else:
    with open(cfg.MAP_PATH, 'rb') as f:
        term_to_cat = pickle.load(f)
    print(f'Map loaded ({len(term_to_cat)-1} terms).')
default_cat = term_to_cat.get('_DEFAULT_', 'FOOD#QUALITY')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'TypeError: Failed to fetch'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.
Map built: 3400 terms. Default: "FOOD#QUALITY"


## 📊 Load & Split All Datasets
Combines all XML files, then splits into **80% train / 10% val / 10% test**.

In [ ]:
def three_way_split(dataset, train_r, val_r, test_r, seed=42):
    """Split a dataset into train/val/test with fixed proportions."""
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6, 'Ratios must sum to 1'
    n      = len(dataset)
    n_test = max(1, int(n * test_r))
    n_val  = max(1, int(n * val_r))
    n_train = n - n_val - n_test
    return random_split(dataset, [n_train, n_val, n_test],
                        generator=torch.Generator().manual_seed(seed))


# ── Aspect Extraction datasets ──────────────────────────────────
print('Loading Aspect Extraction datasets:')
ext_datasets = []
for path in DATASET_PATHS:
    if not os.path.exists(path):
        print(f'  Skipping (not found): {path}')
        continue
    ds = AspectExtractionDataset(path, tokenizer, cfg.MAX_LEN)
    if len(ds) > 0:
        ext_datasets.append(ds)

assert ext_datasets, 'No extraction data loaded — check DATASET_PATHS'
combined_ext = ConcatDataset(ext_datasets)
print(f'Total extraction examples: {len(combined_ext)}')

train_ext, val_ext, test_ext = three_way_split(
    combined_ext, cfg.TRAIN_RATIO, cfg.VAL_RATIO, cfg.TEST_RATIO
)
print(f'Extraction split → train:{len(train_ext)} | val:{len(val_ext)} | test:{len(test_ext)}')

train_loader_ext = DataLoader(train_ext, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader_ext   = DataLoader(val_ext,   batch_size=cfg.BATCH_SIZE)
test_loader_ext  = DataLoader(test_ext,  batch_size=cfg.BATCH_SIZE)


# ── Sentiment datasets ──────────────────────────────────────────
def sentiment_collate(batch):
    sentences = [ex['sentence'] for ex in batch]
    aspects   = [ex['aspect']   for ex in batch]
    labels    = [ex['label']    for ex in batch]
    encoded   = tokenizer(
        sentences, aspects,
        truncation='only_first',
        padding='max_length',
        max_length=cfg.MAX_LEN,
        return_tensors='pt'
    )
    input_ids      = encoded['input_ids']
    attention_mask = encoded['attention_mask']
    sep_id         = tokenizer.sep_token_id
    aspect_mask    = torch.zeros_like(input_ids)
    for i in range(input_ids.size(0)):
        ids = input_ids[i].tolist()
        first_sep = ids.index(sep_id)
        try:
            second_sep = ids.index(sep_id, first_sep + 1)
        except ValueError:
            second_sep = len(ids)
        aspect_mask[i, first_sep + 1 : second_sep] = 1
    return {
        'input_ids':      input_ids,
        'attention_mask': attention_mask,
        'aspect_mask':    aspect_mask,
        'labels':         torch.tensor(labels, dtype=torch.long)
    }


print('\nLoading Sentiment datasets:')
sent_datasets = []
for path in DATASET_PATHS:
    if not os.path.exists(path):
        continue
    ds = SentimentDataset(path, tokenizer, cfg.MAX_LEN)
    if len(ds) > 0:
        sent_datasets.append(ds)

assert sent_datasets, 'No sentiment data loaded — check DATASET_PATHS'
combined_sent = ConcatDataset(sent_datasets)
print(f'Total sentiment examples: {len(combined_sent)}')

train_sent, val_sent, test_sent = three_way_split(
    combined_sent, cfg.TRAIN_RATIO, cfg.VAL_RATIO, cfg.TEST_RATIO
)
print(f'Sentiment split → train:{len(train_sent)} | val:{len(val_sent)} | test:{len(test_sent)}')

train_loader_sent = DataLoader(train_sent, batch_size=cfg.BATCH_SIZE, shuffle=True, collate_fn=sentiment_collate)
val_loader_sent   = DataLoader(val_sent,   batch_size=cfg.BATCH_SIZE, collate_fn=sentiment_collate)
test_loader_sent  = DataLoader(test_sent,  batch_size=cfg.BATCH_SIZE, collate_fn=sentiment_collate)

Loading Aspect Extraction datasets:
  [2016] ABSA16_Restaurants_Train_SB1_v2.xml: 1234 examples loaded, 766 skipped (no spans)
  [2016] ABSA-15_Restaurants_Train_Final.xml: 833 examples loaded, 482 skipped (no spans)
  [2014] Restaurants_Train.xml: 2023 examples loaded, 1021 skipped (no spans)
  [2014] train.xml: 4297 examples loaded, 0 skipped (no spans)
Total extraction examples: 8387
Extraction split → train:6711 | val:838 | test:838

Loading Sentiment datasets:
  [2016] ABSA16_Restaurants_Train_SB1_v2.xml: 2507 sentiment examples
  [2016] ABSA-15_Restaurants_Train_Final.xml: 1654 sentiment examples
  [2014] Restaurants_Train.xml: 3608 sentiment examples
  [2014] train.xml: 11186 sentiment examples
Total sentiment examples: 18955
Sentiment split → train:15165 | val:1895 | test:1895


## 🏋️ Train Aspect Extractor

In [ ]:
model_ext  = AspectExtractorDistilBERT(num_tags=cfg.NUM_TAGS).to(cfg.DEVICE)
opt_ext    = AdamW(model_ext.parameters(), lr=cfg.LEARNING_RATE)
total_ext  = len(train_loader_ext) * cfg.EPOCHS_EXTRACTOR
sched_ext  = get_linear_schedule_with_warmup(
    opt_ext,
    num_warmup_steps=int(cfg.WARMUP_RATIO * total_ext),
    num_training_steps=total_ext
)

best_ext_loss = float('inf')

for epoch in range(cfg.EPOCHS_EXTRACTOR):
    # ── Train ──
    model_ext.train()
    total_loss = 0
    for batch in train_loader_ext:
        ids   = batch['input_ids'].to(cfg.DEVICE)
        mask  = batch['attention_mask'].to(cfg.DEVICE)
        tags  = batch['tag_ids'].to(cfg.DEVICE)
        opt_ext.zero_grad()
        loss, _ = model_ext(ids, mask, tag_ids=tags)
        loss.backward()
        opt_ext.step()
        sched_ext.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader_ext)

    # ── Validate ──
    model_ext.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader_ext:
            ids  = batch['input_ids'].to(cfg.DEVICE)
            mask = batch['attention_mask'].to(cfg.DEVICE)
            tags = batch['tag_ids'].to(cfg.DEVICE)
            _, logits = model_ext(ids, mask, tag_ids=tags)
            preds  = torch.argmax(logits, dim=-1)
            active = mask.view(-1) == 1
            all_preds.extend(preds.view(-1)[active].cpu().numpy())
            all_labels.extend(tags.view(-1)[active].cpu().numpy())

    filtered = [(p, l) for p, l in zip(all_preds, all_labels) if l != -100]
    if filtered:
        pf, lf = zip(*filtered)
        print(f'Epoch {epoch+1} | train loss: {avg_loss:.4f}')
        print(classification_report(lf, pf,
              target_names=list(cfg.ID2TAG.values()), zero_division=0))

    if avg_loss < best_ext_loss:
        best_ext_loss = avg_loss
        torch.save(model_ext.state_dict(), cfg.EXT_PATH)
        print(f'  -> Best extractor saved (loss={best_ext_loss:.4f})')

print('Aspect extractor training complete.')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | train loss: 0.3105
              precision    recall  f1-score   support

           O       0.97      0.96      0.97     16848
    B-ASPECT       0.77      0.86      0.81      1771
    I-ASPECT       0.76      0.68      0.72      1214

    accuracy                           0.94     19833
   macro avg       0.83      0.83      0.83     19833
weighted avg       0.94      0.94      0.94     19833

  -> Best extractor saved (loss=0.3105)
Epoch 2 | train loss: 0.1335
              precision    recall  f1-score   support

           O       0.98      0.96      0.97     16848
    B-ASPECT       0.78      0.88      0.83      1771
    I-ASPECT       0.72      0.79      0.75      1214

    accuracy                           0.94     19833
   macro avg       0.83      0.88      0.85     19833
weighted avg       0.94      0.94      0.94     19833

  -> Best extractor saved (loss=0.1335)
Epoch 3 | train loss: 0.1048
              precision    recall  f1-score   support

           O    

## 🏋️ Train Sentiment Classifier

In [ ]:
model_sent  = AspectAwareDistilBERT(num_labels=3).to(cfg.DEVICE)
opt_sent    = AdamW(model_sent.parameters(), lr=cfg.LEARNING_RATE)
total_sent  = len(train_loader_sent) * cfg.EPOCHS_SENTIMENT
sched_sent  = get_linear_schedule_with_warmup(
    opt_sent,
    num_warmup_steps=int(cfg.WARMUP_RATIO * total_sent),
    num_training_steps=total_sent
)
criterion = nn.CrossEntropyLoss()
best_f1   = 0.0

for epoch in range(cfg.EPOCHS_SENTIMENT):
    # ── Train ──
    model_sent.train()
    total_loss = 0
    for batch in train_loader_sent:
        ids  = batch['input_ids'].to(cfg.DEVICE)
        mask = batch['attention_mask'].to(cfg.DEVICE)
        asp  = batch['aspect_mask'].to(cfg.DEVICE)
        lbl  = batch['labels'].to(cfg.DEVICE)
        opt_sent.zero_grad()
        loss = criterion(model_sent(ids, mask, asp), lbl)
        loss.backward()
        opt_sent.step()
        sched_sent.step()
        total_loss += loss.item()

    # ── Validate ──
    model_sent.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader_sent:
            ids  = batch['input_ids'].to(cfg.DEVICE)
            mask = batch['attention_mask'].to(cfg.DEVICE)
            asp  = batch['aspect_mask'].to(cfg.DEVICE)
            lbl  = batch['labels'].to(cfg.DEVICE)
            preds = torch.argmax(model_sent(ids, mask, asp), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbl.cpu().numpy())

    rep = classification_report(all_labels, all_preds,
                                 target_names=['positive','negative','neutral'],
                                 output_dict=True, zero_division=0)
    f1  = rep['macro avg']['f1-score']
    print(f'Epoch {epoch+1} | loss: {total_loss/len(train_loader_sent):.4f} | val macro-F1: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model_sent.state_dict(), cfg.SENT_PATH)
        print('  -> Best sentiment model saved')

print(f'Sentiment training complete. Best val macro-F1 = {best_f1:.4f}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | loss: 0.7027 | val macro-F1: 0.7854
  -> Best sentiment model saved
Epoch 2 | loss: 0.3952 | val macro-F1: 0.8061
  -> Best sentiment model saved
Epoch 3 | loss: 0.2489 | val macro-F1: 0.8136
  -> Best sentiment model saved
Epoch 4 | loss: 0.1525 | val macro-F1: 0.8103
Epoch 5 | loss: 0.0957 | val macro-F1: 0.8102
Sentiment training complete. Best val macro-F1 = 0.8136


## 📈 Final Test Set Evaluation
Loads the best saved checkpoints and evaluates on the held-out 10% test split.

In [ ]:
# Load best checkpoints
model_ext.load_state_dict(torch.load(cfg.EXT_PATH, map_location=cfg.DEVICE))
model_ext.eval()
model_sent.load_state_dict(torch.load(cfg.SENT_PATH, map_location=cfg.DEVICE))
model_sent.eval()

# ── Aspect Extraction Test Report ──
all_true_tags, all_pred_tags = [], []
with torch.no_grad():
    for batch in test_loader_ext:
        ids  = batch['input_ids'].to(cfg.DEVICE)
        mask = batch['attention_mask'].to(cfg.DEVICE)
        tags = batch['tag_ids'].to(cfg.DEVICE)
        _, logits = model_ext(ids, mask, tag_ids=tags)
        preds = torch.argmax(logits, dim=-1)
        for i in range(ids.size(0)):
            true_seq, pred_seq = [], []
            for j in range(ids.size(1)):
                if mask[i, j] == 0: continue
                gold = tags[i, j].item()
                if gold == -100: continue
                true_seq.append(cfg.ID2TAG[gold])
                pred_seq.append(cfg.ID2TAG[preds[i, j].item()])
            if true_seq:
                all_true_tags.append(true_seq)
                all_pred_tags.append(pred_seq)

print('=' * 55)
print('ASPECT EXTRACTION — TEST SET RESULTS')
print('=' * 55)
print(seq_report(all_true_tags, all_pred_tags, digits=4))

# ── Sentiment Classification Test Report ──
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader_sent:
        ids  = batch['input_ids'].to(cfg.DEVICE)
        mask = batch['attention_mask'].to(cfg.DEVICE)
        asp  = batch['aspect_mask'].to(cfg.DEVICE)
        lbl  = batch['labels'].to(cfg.DEVICE)
        preds = torch.argmax(model_sent(ids, mask, asp), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbl.cpu().numpy())

print('=' * 55)
print('SENTIMENT CLASSIFICATION — TEST SET RESULTS')
print('=' * 55)
print(classification_report(all_labels, all_preds,
      target_names=['positive','negative','neutral'],
      digits=4, zero_division=0))

ASPECT EXTRACTION — TEST SET RESULTS
              precision    recall  f1-score   support

      ASPECT     0.7437    0.8397    0.7888      1759

   micro avg     0.7437    0.8397    0.7888      1759
   macro avg     0.7437    0.8397    0.7888      1759
weighted avg     0.7437    0.8397    0.7888      1759

SENTIMENT CLASSIFICATION — TEST SET RESULTS
              precision    recall  f1-score   support

    positive     0.8505    0.8838    0.8669       792
    negative     0.8370    0.8505    0.8437       495
     neutral     0.8295    0.7763    0.8020       608

    accuracy                         0.8406      1895
   macro avg     0.8390    0.8369    0.8375      1895
weighted avg     0.8403    0.8406    0.8400      1895



## 🔧 Clause Splitter + Inference Pipeline

In [ ]:
def split_into_clauses(sentence):
    doc     = nlp(sentence)
    clauses = []
    start   = 0
    for token in doc:
        if token.pos_ == 'CCONJ' and token.dep_ == 'cc':
            if token.i > start:
                clauses.append(doc[start:token.i].text.strip())
            start = token.i + 1
        elif token.dep_ == 'mark' and token.head.pos_ in ('VERB', 'AUX'):
            if token.i > start:
                clauses.append(doc[start:token.i].text.strip())
            start = token.i + 1
    if start < len(doc):
        clauses.append(doc[start:].text.strip())
    clauses = [c for c in clauses if c.strip()]
    return clauses if clauses else [sentence]


def extract_aspects_from_fragment(fragment):
    enc   = tokenizer(fragment, truncation=True, padding='max_length',
                      max_length=cfg.MAX_LEN, return_tensors='pt')
    ids   = enc['input_ids'].to(cfg.DEVICE)
    mask  = enc['attention_mask'].to(cfg.DEVICE)
    with torch.no_grad():
        preds = torch.argmax(model_ext(ids, mask), dim=-1).squeeze(0).cpu().numpy()
    seq_len = mask.sum().item()
    tokens  = tokenizer.convert_ids_to_tokens(ids[0][:seq_len])
    preds   = preds[:seq_len]
    aspects, current = [], []
    for token, tag in zip(tokens, preds):
        if token.startswith('##') and current and tag != cfg.TAG2ID['O']:
            current.append(token)
        elif tag == cfg.TAG2ID['B-ASPECT']:
            if current:
                aspects.append(''.join(t.lstrip('#') if t.startswith('##') else t for t in current))
            current = [token]
        elif tag == cfg.TAG2ID['I-ASPECT'] and current:
            current.append(token)
        else:
            if current:
                aspects.append(''.join(t.lstrip('#') if t.startswith('##') else t for t in current))
            current = []
    if current:
        aspects.append(''.join(t.lstrip('#') if t.startswith('##') else t for t in current))
    return aspects


def classify_sentiment(fragment, aspect_term):
    category = term_to_cat.get(aspect_term.lower(), default_cat)
    enc      = tokenizer(fragment, category, truncation='only_first',
                         padding='max_length', max_length=cfg.MAX_LEN,
                         return_tensors='pt')
    ids  = enc['input_ids'].to(cfg.DEVICE)
    mask = enc['attention_mask'].to(cfg.DEVICE)
    sep_id = tokenizer.sep_token_id
    id_list = ids[0].tolist()
    asp_mask = torch.zeros_like(ids)
    first_sep = id_list.index(sep_id)
    try:
        second_sep = id_list.index(sep_id, first_sep + 1)
    except ValueError:
        second_sep = len(id_list)
    asp_mask[0, first_sep + 1 : second_sep] = 1
    asp_mask = asp_mask.to(cfg.DEVICE)
    with torch.no_grad():
        pred = torch.argmax(model_sent(ids, mask, asp_mask), dim=1).item()
    return cfg.ID2SENTIMENT[pred]


def analyze(sentence):
    results = []
    for frag in split_into_clauses(sentence):
        for asp in extract_aspects_from_fragment(frag):
            results.append((asp, classify_sentiment(frag, asp)))
    return results


# Quick tests
tests = [
    "The food was great but the service was terrible.",
    "Waiter was rude and the pasta was overcooked."
]
for t in tests:
    print(f'Input : {t}')
    print(f'Output: {analyze(t)}')
    print()

Input : The food was great but the service was terrible.
Output: [('food', 'positive'), ('service', 'negative')]

Input : Waiter was rude and the pasta was overcooked.
Output: [('waiter', 'negative'), ('pasta', 'negative')]

